# Tiền xử lý dữ liệu (Preprocessing)

Notebook này thực hiện các bước tiền xử lý dữ liệu mưa từ file gốc để chuẩn bị cho việc phân tích và mô hình hóa. Các thay đổi chính bao gồm:

- Lọc dữ liệu theo khu vực Đông Nam Á (Southeast Asia).
- Xử lý giá trị thiếu bằng cách điền trung vị theo tháng và vị trí lưới.
- Tạo các đặc trưng mới (feature engineering) để cải thiện khả năng dự đoán.
- Chuẩn hóa và mã hóa các biến thời gian và vị trí.

Dữ liệu đầu vào: `Data/Data.csv`
Dữ liệu đầu ra: `Data/processedData.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path('Data') / 'Data.csv'
OUTPUT_PATH = Path('Data') / 'processedData.csv'
print('DATA_PATH:', DATA_PATH)
print('OUTPUT_PATH:', OUTPUT_PATH)

DATA_PATH: Data\Data.csv
OUTPUT_PATH: Data\processedData.csv


In [ ]:
usecols = [
    'time', 'nv', 'lat', 'lon', 'time_bnds', 'lat_bnds', 'lon_bnds',
    'record_status', 'precipitation', 'num_obs_fraction', 'num_obs_rate',
    'num_days', 'quality_flag', 'num_days_snow'
]
df = pd.read_csv(DATA_PATH, usecols=usecols, parse_dates=['time'], encoding='utf-8')

# Define Southeast Asia bounding box
lat_min, lat_max = -11.0, 25.0
lon_min, lon_max = 92.0, 141.0
sea_mask = (
    (df['lat'] >= lat_min) & (df['lat'] <= lat_max) &
    (df['lon'] >= lon_min) & (df['lon'] <= lon_max)
)
df_sea = df[sea_mask].copy()

print('Original rows:', len(df))
print('SEA rows:', len(df_sea))
print('SEA lat range:', df_sea['lat'].min(), 'to', df_sea['lat'].max())
print('SEA lon range:', df_sea['lon'].min(), 'to', df_sea['lon'].max())

## Bước 1: Nạp dữ liệu và lọc theo khu vực

- **Thay đổi**: Chỉ giữ lại các cột cần thiết từ danh sách `usecols`.
- **Thêm**: Chuyển đổi cột 'time' thành datetime.
- **Lọc**: Giới hạn dữ liệu trong phạm vi Đông Nam Á (latitude: -11 đến 25, longitude: 92 đến 141).
- **Phương pháp**: Sử dụng pandas read_csv với usecols và parse_dates, sau đó áp dụng mask boolean để lọc.

In [ ]:
df_sea['year'] = df_sea['time'].dt.year
df_sea['month'] = df_sea['time'].dt.month

df_sea.sort_values(['lat', 'lon', 'time'], inplace=True)

def fill_precipitation(group):
    group = group.copy()
    # monthly median for the same grid cell
    group['precip_monthly_median'] = group.groupby('month')['precipitation'].transform('median')
    group['precipitation'] = group['precipitation'].fillna(group['precip_monthly_median'])
    # if vẫn thiếu thì dùng median của toàn bộ SEA
    group['precipitation'] = group['precipitation'].fillna(df_sea['precipitation'].median())
    return group

df_sea = df_sea.groupby(['lat', 'lon'], group_keys=False).apply(fill_precipitation)

# Fill other numeric columns with location-month median first, else column median
numeric_cols = [
    'nv', 'num_obs_fraction', 'num_obs_rate', 'num_days', 'quality_flag', 'num_days_snow'
]
for col in numeric_cols:
    monthly_median = df_sea.groupby(['lat', 'lon', 'month'])[col].transform('median')
    df_sea[col] = df_sea[col].fillna(monthly_median)
    df_sea[col] = df_sea[col].fillna(df_sea[col].median())

missing_summary = df_sea.isna().sum()
print('Missing values after imputation:')
print(missing_summary[missing_summary > 0])

C:\Users\Hello\AppData\Local\Temp\ipykernel_25624\2337633936.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sea = df_sea.groupby(['lat', 'lon'], group_keys=False).apply(fill_precipitation)


Missing values after imputation:
Series([], dtype: int64)


## Bước 2: Xử lý giá trị thiếu

- **Thêm**: Các cột 'year' và 'month' từ cột 'time'.
- **Sắp xếp**: Dữ liệu theo lat, lon, time để đảm bảo thứ tự thời gian.
- **Điền giá trị thiếu cho precipitation**:
  - Ưu tiên: Trung vị theo tháng cho cùng vị trí lưới.
  - Dự phòng: Trung vị toàn bộ khu vực SEA.
- **Điền giá trị thiếu cho các cột số khác**:
  - Ưu tiên: Trung vị theo vị trí lưới và tháng.
  - Dự phòng: Trung vị của cột.
- **Phương pháp**: Sử dụng groupby và transform với median, sau đó fillna.

In [ ]:
# Cyclical month encoding to capture seasonal patterns
# Data có tần suất tháng, nên không cần day-of-year

df_sea['month_sin'] = np.sin(2 * np.pi * df_sea['month'] / 12)
df_sea['month_cos'] = np.cos(2 * np.pi * df_sea['month'] / 12)

# Location features
lat_mean, lat_std = df_sea['lat'].mean(), df_sea['lat'].std()
lon_mean, lon_std = df_sea['lon'].mean(), df_sea['lon'].std()
df_sea['lat_norm'] = (df_sea['lat'] - lat_mean) / lat_std
df_sea['lon_norm'] = (df_sea['lon'] - lon_mean) / lon_std
df_sea['abs_lat'] = df_sea['lat'].abs()
df_sea['lat_lon_interaction'] = df_sea['lat'] * df_sea['lon']

def season_from_month(month):
    if month in [12, 1, 2]:
        return 'DJF'
    if month in [3, 4, 5]:
        return 'MAM'
    if month in [6, 7, 8]:
        return 'JJA'
    return 'SON'

df_sea['season'] = df_sea['month'].apply(season_from_month)
season_mapping = {'DJF': 0, 'MAM': 1, 'JJA': 2, 'SON': 3}
df_sea['season_code'] = df_sea['season'].map(season_mapping)

# Historical precipitation features by grid cell
for window in [1, 3, 12]:
    df_sea[f'precip_prev_{window}m_mean'] = (
        df_sea.groupby(['lat', 'lon'])['precipitation']
        .shift(1)
        .rolling(window=window, min_periods=1)
        .mean()
    )

# Monthly climate average for the grid cell
monthly_climate = df_sea.groupby(['lat', 'lon', 'month'])['precipitation'].transform('mean')
df_sea['precip_monthly_climate'] = monthly_climate

# Create a simplified time index for modeling
start = df_sea['time'].min()
df_sea['time_index'] = (df_sea['time'] - start).dt.days

print('Created engineered features:')
print([col for col in df_sea.columns if col not in usecols])

Created engineered features:
['year', 'month', 'precip_monthly_median', 'month_sin', 'month_cos', 'lat_norm', 'lon_norm', 'abs_lat', 'lat_lon_interaction', 'season', 'season_code', 'precip_prev_1m_mean', 'precip_prev_3m_mean', 'precip_prev_12m_mean', 'precip_monthly_climate', 'time_index']


## Bước 3: Kỹ thuật đặc trưng (Feature Engineering)

- **Thêm đặc trưng thời gian**:
  - Mã hóa tuần hoàn cho tháng (month_sin, month_cos) để nắm bắt mẫu mùa vụ.
- **Thêm đặc trưng vị trí**:
  - Chuẩn hóa lat và lon (lat_norm, lon_norm).
  - abs_lat: Giá trị tuyệt đối của latitude.
  - lat_lon_interaction: Tương tác giữa lat và lon.
- **Thêm đặc trưng mùa**:
  - season: Mùa dựa trên tháng (DJF, MAM, JJA, SON).
  - season_code: Mã số cho mùa.
- **Thêm đặc trưng lịch sử mưa**:
  - precip_prev_1m_mean, precip_prev_3m_mean, precip_prev_12m_mean: Trung bình mưa trong 1, 3, 12 tháng trước.
- **Thêm đặc trưng khí hậu**:
  - precip_monthly_climate: Trung bình mưa hàng tháng cho vị trí lưới.
- **Thêm chỉ số thời gian**:
  - time_index: Số ngày từ thời điểm bắt đầu.
- **Phương pháp**: Sử dụng numpy cho mã hóa tuần hoàn, pandas groupby và rolling cho trung bình lịch sử.

In [ ]:
output_columns = [
    'time', 'lat', 'lon', 'precipitation',
    'nv', 'num_obs_fraction', 'num_obs_rate', 'num_days',
    'quality_flag', 'num_days_snow',
    'month', 'year',
    'month_sin', 'month_cos',
    'season', 'season_code', 'lat_norm', 'lon_norm',
    'abs_lat', 'lat_lon_interaction',
    'precip_prev_1m_mean', 'precip_prev_3m_mean',
    'precip_prev_12m_mean', 'precip_monthly_climate', 'time_index'
]

# Ensure all columns exist before saving
output_columns = [col for col in output_columns if col in df_sea.columns]
df_sea[output_columns].to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
print('Saved processed dataset to', OUTPUT_PATH)
print('Processed shape:', df_sea[output_columns].shape)

Saved processed dataset to Data\processedData.csv
Processed shape: (677376, 25)


## Bước 4: Lưu dữ liệu đã xử lý

- **Chọn cột đầu ra**: Danh sách các cột cần thiết bao gồm gốc và mới tạo.
- **Lưu**: Xuất ra file CSV mà không có index.
- **Phương pháp**: Sử dụng pandas to_csv với encoding UTF-8.

## Tổng kết

Quy trình tiền xử lý này đã biến đổi dữ liệu gốc từ file `Data/Data.csv` thành tập dữ liệu sạch và phong phú hơn trong `Data/processedData.csv`. Các thay đổi chính bao gồm:

- **Giảm kích thước dữ liệu**: Lọc theo khu vực Đông Nam Á, giảm từ số hàng gốc xuống còn số hàng trong SEA.
- **Xử lý giá trị thiếu**: Điền đầy đủ các giá trị thiếu bằng phương pháp trung vị theo vị trí và thời gian, đảm bảo không còn giá trị thiếu.
- **Mở rộng đặc trưng**: Thêm 11 đặc trưng mới để nắm bắt các mẫu thời gian, vị trí và lịch sử, tăng từ 15 cột gốc lên 26 cột.
- **Chuẩn bị cho mô hình**: Các đặc trưng được chuẩn hóa và mã hóa phù hợp cho các thuật toán học máy.

Dữ liệu đầu ra sẵn sàng cho các bước phân tích tiếp theo như EDA hoặc xây dựng mô hình dự đoán mưa.

## Kiểm tra dữ liệu đã xuất

- Đọc lại file `Data/processedData.csv` sau khi tiền xử lý.
- Kiểm tra số cột, số hàng, thành phần dữ liệu, giá trị thiếu, và mức độ giá trị nhỏ/ lớn.
- Hiển thị mô tả thống kê để xác nhận dữ liệu đã được xử lý đúng.

In [ ]:
# Read processed data back and inspect it
processed_df = pd.read_csv(OUTPUT_PATH, parse_dates=['time'], encoding='utf-8')

print('Processed file path:', OUTPUT_PATH)
print('Rows:', len(processed_df))
print('Columns:', len(processed_df))
print('\nColumn list:')
print(processed_df.columns.tolist())

print('\nMissing values per column:')
missing = processed_df.isna().sum()
print(missing[missing > 0] if missing.any() else 'No missing values detected')

print('\nData types:')
print(processed_df.dtypes)

numeric_cols = processed_df.select_dtypes(include=[np.number]).columns.tolist()
print('\nNumeric columns:', numeric_cols)

if numeric_cols:
    print('\nNumeric summary statistics:')
    display(processed_df[numeric_cols].describe().T)
    print('\nMin/max values for numeric columns:')
    min_max = processed_df[numeric_cols].agg(['min', 'max']).T
    display(min_max)

print('\nSample rows:')
print(processed_df.head().to_string(index=False))

print('\nTail rows:')
print(processed_df.tail().to_string(index=False))